# Phase 5 prime bundle-first n=10 controls (Colab)

This notebook runs the existing `experiments/44_phase5_prime_bundle_first.py` harness on Colab. It is a Phase 5 prime diagnostic, not a Phase 5 graduation run.

The default path avoids Google Drive mount and writes results under `/content`. Download result JSONs from the final cell if you want to preserve them.

Planned sequence:

1. Clone `Dypatterson/Neuro-AI` and check out `phase5-m1-role-energy-stack`.
2. Run a tiny smoke.
3. Run the hard-cell n=10 control rerun: `D=4096`, `N=512`, `K_roles=16`, `cue_noise=0.15`, `scene_token=1`, skewed co-occurrence.
4. Run a candidate-only n=10 grid across K, N, cue noise, scene-token, and co-occurrence.
5. Keep the full all-controls matrix as an explicit opt-in cell because it is much larger.


In [ ]:
# 1. Clone the repo and check out the active branch.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git /content/Neuro-AI
%cd /content/Neuro-AI
!git checkout phase5-m1-role-energy-stack
!git log --oneline -1

# Verify the Phase 5 prime harness exists on this branch.
!test -f experiments/44_phase5_prime_bundle_first.py
!grep -n "bundle-first structural-memory" experiments/44_phase5_prime_bundle_first.py | head -1

In [ ]:
# 2. Runtime and harness sanity.
import json, os, subprocess, time
from pathlib import Path

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("device", DEVICE)

if DEVICE == "cuda":
    !nvidia-smi --query-gpu=name,memory.total --format=csv

!PYTHONPATH=src:. python -m py_compile experiments/44_phase5_prime_bundle_first.py

In [ ]:
# 3. Tiny smoke. Expected: candidate > controls, positives at 1.0 on this small cell.
!PYTHONPATH=src:. python experiments/44_phase5_prime_bundle_first.py \
  --Ds 128 --Ns 8 --K_roles 2 --cue_noise 0.0 \
  --seeds 17 --n_queries 8 --C_codebook 32 \
  --conditions candidate random_role shuffled_role perfect_cue bundle_positive content_cleanup_positive \
  --scene_token 0 --cooccurrence uniform --device cpu \
  --out /content/phase5_prime_smoke.json

In [ ]:
# 4. Hard-cell n=10 controls. This reproduces the most important Report 069 cell.
HARD_CELL_OUT = "/content/phase5_prime_bundle_hard_cell_n10.json"
!PYTHONPATH=src:. python experiments/44_phase5_prime_bundle_first.py \
  --Ds 4096 --Ns 512 --K_roles 16 --cue_noise 0.15 \
  --seeds 17 11 23 1 2 3 5 7 13 29 \
  --n_queries 512 --C_codebook 1024 \
  --conditions candidate random_role shuffled_role perfect_cue bundle_positive content_cleanup_positive \
  --scene_token 1 --cooccurrence skewed --device {DEVICE} \
  --out {HARD_CELL_OUT}

In [ ]:
# 5. Summarize the hard-cell controls.
def summarize_payload(path):
    payload = json.loads(Path(path).read_text())
    rows = []
    for key, agg in payload["aggregates"].items():
        rows.append((key, agg))
    for key, agg in sorted(rows):
        print(
            f"{key}\n"
            f"  top1={agg['top1_mean']:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] "
            f"scene_tix={agg['scene_tix_rate']:.4f} content_tix={agg['content_tix_rate']:.4f} "
            f"n={agg['n_total']}"
        )
    return payload

hard_payload = summarize_payload(HARD_CELL_OUT)

In [ ]:
# 6. Candidate-only n=10 grid across supported Phase 5 prime axes.
# This is much cheaper than the full all-controls grid and identifies boundary cells.
CANDIDATE_GRID_OUT = "/content/phase5_prime_candidate_grid_n10.json"
!PYTHONPATH=src:. python experiments/44_phase5_prime_bundle_first.py \
  --Ds 4096 --Ns 16 32 64 128 256 512 \
  --K_roles 2 4 8 16 \
  --cue_noise 0.0 0.05 0.10 0.15 \
  --seeds 17 11 23 1 2 3 5 7 13 29 \
  --n_queries 256 --C_codebook 1024 \
  --conditions candidate \
  --scene_token 0 1 --cooccurrence uniform skewed --device {DEVICE} \
  --out {CANDIDATE_GRID_OUT}

In [ ]:
# 7. Summarize candidate grid: worst and best cells by top1.
candidate_payload = json.loads(Path(CANDIDATE_GRID_OUT).read_text())
rows = sorted(
    ((agg["top1_mean"], key, agg) for key, agg in candidate_payload["aggregates"].items()),
    key=lambda x: x[0],
)

print("Worst 20 candidate cells:")
for top1, key, agg in rows[:20]:
    print(f"{top1:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] scene={agg['scene_tix_rate']:.4f} content={agg['content_tix_rate']:.4f} :: {key}")

print("\nBest 20 candidate cells:")
for top1, key, agg in rows[-20:]:
    print(f"{top1:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] scene={agg['scene_tix_rate']:.4f} content={agg['content_tix_rate']:.4f} :: {key}")

In [ ]:
# 8. Optional full all-controls matrix. This is large: run only after the hard-cell and candidate grid are sane.
RUN_FULL_MATRIX = False

if RUN_FULL_MATRIX:
    FULL_MATRIX_OUT = "/content/phase5_prime_full_controls_grid_n10.json"
    cmd = [
        "python", "experiments/44_phase5_prime_bundle_first.py",
        "--Ds", "4096",
        "--Ns", "16", "32", "64", "128", "256", "512",
        "--K_roles", "2", "4", "8", "16",
        "--cue_noise", "0.0", "0.05", "0.10", "0.15",
        "--seeds", "17", "11", "23", "1", "2", "3", "5", "7", "13", "29",
        "--n_queries", "512",
        "--C_codebook", "1024",
        "--conditions", "candidate", "random_role", "shuffled_role", "perfect_cue", "bundle_positive", "content_cleanup_positive",
        "--scene_token", "0", "1",
        "--cooccurrence", "uniform", "skewed",
        "--device", DEVICE,
        "--out", FULL_MATRIX_OUT,
    ]
    env = dict(os.environ, PYTHONPATH="src:.")
    subprocess.run(cmd, check=True, env=env)
    print("wrote", FULL_MATRIX_OUT)
else:
    print("Full all-controls matrix skipped. Set RUN_FULL_MATRIX = True to run it.")

In [ ]:
# 9. Download result JSONs from the Colab runtime.
from google.colab import files

for path in [
    "/content/phase5_prime_smoke.json",
    "/content/phase5_prime_bundle_hard_cell_n10.json",
    "/content/phase5_prime_candidate_grid_n10.json",
]:
    if Path(path).exists():
        print("downloading", path)
        files.download(path)
